In [1]:
# 1. 라이브러리 및 폰트 설치 (최초 1회 실행 시 약간의 시간 소요)
!apt-get update -qq
!apt-get install fonts-nanum* -qq
!pip install -q ultralytics opencv-python-headless Pillow

import cv2
import math
import numpy as np
from collections import deque
from google.colab import files
from PIL import ImageFont, ImageDraw, Image
from ultralytics import YOLO

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fonts-nanum.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Selecting previously unselected package fonts-nanum-coding.
Preparing to unpack .../fonts-nanum-coding_2.5-3_all.deb ...
Unpacking fonts-nanum-coding (2.5-3) ...
Selecting previously unselected package fonts-nanum-eco.
Preparing to unpack .../fonts-nanum-eco_1.000-7_all.deb ...
Unpacking fonts-nanum-eco (1.000-7) ...
Selecting previously unselected package fonts-nanum-extra.
Preparing to unpack .../fonts-nanum-extra_20200506-1_all.deb ...
Unpacking fonts-nanum-extra (20200506-1) ...
Setting up fonts-nanum-extra (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Setting up fo

In [2]:


# 2. YOLOv8 Pose 모델 불러오기 (가볍고 빠른 nano 모델)
model = YOLO('yolov8n-pose.pt')

# 3. 분석할 영상 업로드
print("면접 영상 파일을 업로드해주세요 (.mp4, .avi 등)")
uploaded = files.upload()
input_video_name = list(uploaded.keys())[0]
output_video_name = 'interview_pose_feedback_' + input_video_name

# 4. 영상 처리 준비
cap = cv2.VideoCapture(input_video_name)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_name, fourcc, fps, (width, height))

# 폰트 세팅
fontpath = "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf"
font = ImageFont.truetype(fontpath, 35)

# 💡 [핵심] 흔들림 감지를 위한 데이터 저장소 (약 1초 분량의 데이터 기록)
history_len = int(fps) if fps > 0 else 30
shoulder_widths = deque(maxlen=history_len)
wrist_movements = deque(maxlen=history_len)
knee_y_movements = deque(maxlen=history_len)

prev_lw, prev_rw = None, None
prev_lk, prev_rk = None, None

print(f"\n'{input_video_name}' 신체 흔들림 분석 시작...")

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    frame_count += 1
    if frame_count % 30 == 0: print(f"{frame_count} 프레임 처리 완료")

    # YOLO 모델로 뼈대 추출 (화면에 관절을 그리지 않고 데이터만 가져옴)
    results = model(frame, verbose=False)

    # 한글 출력을 위해 PIL 이미지로 변환
    frame_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(frame_pil)

    warnings = [] # 이번 프레임에 띄울 경고 메시지들

    if len(results[0].keypoints) > 0 and results[0].keypoints.has_visible:
        # 첫 번째 사람의 관절 데이터 추출 [17개 관절, (x, y, 신뢰도)]
        kpts = results[0].keypoints.data[0].cpu().numpy()

        # 관절 인덱스: 5(왼어깨), 6(오른어깨), 9(왼손목), 10(오른손목), 13(왼무릎), 14(오른무릎)

        # --- 1. 상체 앞뒤 흔들림 감지 ---
        if kpts[5][2] > 0.5 and kpts[6][2] > 0.5:
            s_width = abs(kpts[5][0] - kpts[6][0])
            shoulder_widths.append(s_width)

            if len(shoulder_widths) == history_len:
                # 어깨 너비의 표준편차(변화량)가 일정 수치 이상이면 흔들림으로 간주
                if np.std(shoulder_widths) > 5.0:
                    warnings.append("⚠️ 몸을 앞뒤로 흔들고 있습니다.")

        # --- 2. 손 꼼지락 감지 ---
        current_wrist_move = 0
        if kpts[9][2] > 0.5: # 왼손목
            if prev_lw is not None:
                current_wrist_move += math.hypot(kpts[9][0] - prev_lw[0], kpts[9][1] - prev_lw[1])
            prev_lw = (kpts[9][0], kpts[9][1])

        if kpts[10][2] > 0.5: # 오른손목
            if prev_rw is not None:
                current_wrist_move += math.hypot(kpts[10][0] - prev_rw[0], kpts[10][1] - prev_rw[1])
            prev_rw = (kpts[10][0], kpts[10][1])

        wrist_movements.append(current_wrist_move)
        if len(wrist_movements) == history_len:
            # 1초 동안 손목의 평균 이동량이 일정 수치 이상이면 산만한 것으로 간주
            if np.mean(wrist_movements) > 8.0:
                warnings.append("⚠️ 손을 너무 많이 움직입니다.")

        # --- 3. 다리 떨기 감지 ---
        current_knee_move = 0
        if kpts[13][2] > 0.5: # 왼무릎 (Y축 좌표만 확인)
            if prev_lk is not None:
                current_knee_move += abs(kpts[13][1] - prev_lk)
            prev_lk = kpts[13][1]

        if kpts[14][2] > 0.5: # 오른무릎
            if prev_rk is not None:
                current_knee_move += abs(kpts[14][1] - prev_rk)
            prev_rk = kpts[14][1]

        knee_y_movements.append(current_knee_move)
        if len(knee_y_movements) == history_len:
            # 1초 동안 무릎이 위아래로 움직인 평균 거리가 일정 수치 이상이면 다리 떨기로 간주
            if np.mean(knee_y_movements) > 3.0:
                warnings.append("⚠️ 다리를 떨고 있습니다.")

    # 경고 메시지를 화면 좌측 하단에 차례대로 출력
    start_y = height - 150
    for i, w_text in enumerate(warnings):
        draw.text((30, start_y + (i * 50)), w_text, font=font, fill=(255, 50, 50))

    # PIL -> OpenCV 복구 및 저장
    frame = cv2.cvtColor(np.array(frame_pil), cv2.COLOR_RGB2BGR)
    out.write(frame)

cap.release()
out.release()
print(f"\n✅ 분석 완료! 파일명: {output_video_name}")
files.download(output_video_name)

면접 영상 파일을 업로드해주세요 (.mp4, .avi 등)


Saving bad_test.mp4 to bad_test.mp4

'bad_test.mp4' 신체 흔들림 분석 시작...
30 프레임 처리 완료
60 프레임 처리 완료
90 프레임 처리 완료
120 프레임 처리 완료
150 프레임 처리 완료
180 프레임 처리 완료
210 프레임 처리 완료
240 프레임 처리 완료
270 프레임 처리 완료
300 프레임 처리 완료
330 프레임 처리 완료
360 프레임 처리 완료
390 프레임 처리 완료
420 프레임 처리 완료
450 프레임 처리 완료
480 프레임 처리 완료
510 프레임 처리 완료
540 프레임 처리 완료
570 프레임 처리 완료
600 프레임 처리 완료

✅ 분석 완료! 파일명: interview_pose_feedback_bad_test.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>